# Fashion Recommendation Prediction
## Problem Statement
Predict whether a customer recommends a product using review text, age, and product category.

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

import joblib

## Data Loading

In [15]:
import pandas as pd

df = pd.read_csv('starter/data/reviews.csv')  # ← your correct path
df.head()

,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


In [16]:
def load_data(path):
    return pd.read_csv(path)

def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    print(classification_report(y_test, preds))

## Data Understanding

In [17]:
# Step 2: Understand Columns

print("Columns:")
print(df.columns)

print("\nTarget Column Value Counts:")
print(df['Recommended IND'].value_counts())

Columns:
Index(['Clothing ID', 'Age', 'Title', 'Review Text', 'Positive Feedback Count',
       'Division Name', 'Department Name', 'Class Name', 'Recommended IND'],
      dtype='str')

Target Column Value Counts:
Recommended IND
1    15053
0     3389
Name: count, dtype: int64


In [18]:
text_feature = 'Review Text'

numerical_features = ['Age', 'Positive Feedback Count']

categorical_features = ['Division Name', 'Department Name', 'Class Name']

target = 'Recommended IND'

## Train-Test Split

In [19]:
# Step 3: Train-Test Split

from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (14753, 8)
Test shape: (3689, 8)


## Model Pipeline

In [ ]:
# Step 4: Build Pipeline (FIXED)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# Text pipeline (NO lambda, NO FunctionTransformer)
text_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000))
])

# Numerical pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine all
preprocessor = ColumnTransformer([
    ('text', text_pipeline, text_feature),   
    ('num', num_pipeline, numerical_features),
    ('cat', cat_pipeline, categorical_features)
])

# Full pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

## Model Training

In [21]:
# Train model

model_pipeline.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [22]:
# Step 5: Evaluation

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predictions
y_pred = model_pipeline.predict(X_test)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Full report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.885605855245324
Precision: 0.8984302862419206
Recall: 0.9694453669877118
F1 Score: 0.9325878594249202

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.51      0.62       678
           1       0.90      0.97      0.93      3011

    accuracy                           0.89      3689
   macro avg       0.84      0.74      0.78      3689
weighted avg       0.88      0.89      0.88      3689


Confusion Matrix:

[[ 348  330]
 [  92 2919]]


## Hyperparameter Tuning

In [23]:
# Step 6: Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV

param_grid = {
    # Text tuning
    'preprocessor__text__tfidf__max_features': [3000, 5000],
    'preprocessor__text__tfidf__ngram_range': [(1,1), (1,2)],
    
    # Model tuning
    'classifier__C': [0.1, 1, 10]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1
)

# Fit on training data
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Parameters: {'classifier__C': 1, 'preprocessor__text__tfidf__max_features': 3000, 'preprocessor__text__tfidf__ngram_range': (1, 2)}
Best CV Score: 0.9275204469891287


In [24]:
# Best model
best_model = grid_search.best_estimator_

# Predict again
y_pred_best = best_model.predict(X_test)

from sklearn.metrics import classification_report

print("\nTuned Model Report:\n")
print(classification_report(y_test, y_pred_best))


Tuned Model Report:

              precision    recall  f1-score   support

           0       0.79      0.52      0.62       678
           1       0.90      0.97      0.93      3011

    accuracy                           0.89      3689
   macro avg       0.84      0.74      0.78      3689
weighted avg       0.88      0.89      0.88      3689



In [25]:
# Step 7: Save Model

import joblib

joblib.dump(best_model, 'model.pkl')

print("Model saved as model.pkl")

PicklingError: Can't pickle <function <lambda> at 0x00000174B09E3E20>: it's not found as __main__.<lambda>

In [ ]:
# Load model and test

loaded_model = joblib.load('model.pkl')

sample_pred = loaded_model.predict(X_test[:5])

print("Sample predictions:", sample_pred)

Sample predictions: [1 1 1 1 1]


In [ ]:
# Inference Example

sample = X_test.iloc[:2]

print("Sample Input:")
display(sample)

print("Predictions:")
print(best_model.predict(sample))

# Fashion Recommendation Prediction (ML Pipeline)

## 📌 Problem

Predict whether a customer recommends a product based on:

* Review text
* Age
* Product category

## 📊 Dataset

Women's Clothing E-Commerce Reviews dataset.

## ⚙️ Approach

* Built end-to-end ML pipeline using:

  * TF-IDF for text
  * OneHotEncoding for categorical data
  * StandardScaler for numerical data
* Used Logistic Regression as model
* Applied GridSearchCV for hyperparameter tuning

## 🚀 Model Pipeline

Pipeline includes:

* Preprocessing (text, numerical, categorical)
* Feature engineering
* Model training

## 📈 Evaluation Metrics

* Accuracy
* Precision
* Recall
* F1 Score

## 💾 Model Saving

Final model saved as `model.pkl`

## 📁 Project Structure

DSND-PIPELINES-PROJECT
│
├── notebook.ipynb
├── model.pkl
├── requirements.txt
├── README.md
└── data/

## 🧠 Key Learnings

* End-to-end ML pipeline design
* NLP feature engineering
* Hyperparameter tuning
* Model evaluation
